In [6]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv
from torch.nn import BatchNorm1d

# 1. Load Dataset and Generate Masks
# Flickr doesn't have default masks, so we split it: 70% train, 10% val, 20% test
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

print(f'Dataset: {dataset}')
print(f'Nodes: {data.num_nodes}, Edges: {data.num_edges}')
print(f'Features: {dataset.num_node_features}, Classes: {dataset.num_classes}\n')

class UpgradedGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # 1. Use SAGEConv instead of GCNConv for social networks
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        # 2. Add Batch Normalization to stabilize training
        self.bn1 = BatchNorm1d(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x) # Apply normalization before activation
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# 3. Increase model capacity (change 64 to 256)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UpgradedGNN(dataset.num_node_features, 256, dataset.num_classes).to(device)
# 3. Send the data to the device (THIS IS THE MISSING LINE)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

from torch_geometric.loader import NeighborLoader

# This loader samples 25 neighbors for the first GAT layer, 
# and 10 neighbors for the second GAT layer.
train_loader = NeighborLoader(
    data,
    num_neighbors=[25, 10], 
    batch_size=1024, # Train on 1024 nodes at a time
    input_nodes=data.train_mask,
    shuffle=True
)

# 4. Define Training and Testing Functions
def train():
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index)
        
        # NeighborLoader puts the target nodes at the beginning of the batch.
        # We only calculate loss for these target nodes, not the sampled neighbors.
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size])
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
        
    # Return the average loss for the epoch
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)  # Get the predicted class (highest probability)
    
    accs = []
    # Calculate accuracy for train, val, and test sets
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (pred[mask] == data.y[mask]).sum().item()
        acc = correct / mask.sum().item()
        accs.append(acc)
    return accs

# 5. Execute the Training Loop
print("Starting training...")
for epoch in range(1, 201):  # 100 epochs for GCn, 300 epochs for grpahsage
    loss = train()
    train_acc, val_acc, test_acc = test()
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

# Final Output
print(f'\nTraining Complete!')
print(f'Final Test Accuracy: {test_acc:.4f}')

Dataset: Flickr()
Nodes: 89250, Edges: 899756
Features: 500, Classes: 7

Starting training...
Epoch: 010, Loss: 1.3375, Train Acc: 0.5174, Val Acc: 0.5007
Epoch: 020, Loss: 1.3200, Train Acc: 0.5315, Val Acc: 0.5077
Epoch: 030, Loss: 1.2935, Train Acc: 0.5661, Val Acc: 0.5252
Epoch: 040, Loss: 1.3089, Train Acc: 0.5340, Val Acc: 0.4919
Epoch: 050, Loss: 1.2924, Train Acc: 0.5557, Val Acc: 0.5155
Epoch: 060, Loss: 1.2682, Train Acc: 0.5703, Val Acc: 0.5246
Epoch: 070, Loss: 1.2803, Train Acc: 0.5532, Val Acc: 0.5090
Epoch: 080, Loss: 1.2880, Train Acc: 0.5713, Val Acc: 0.5236
Epoch: 090, Loss: 1.2927, Train Acc: 0.5611, Val Acc: 0.5180
Epoch: 100, Loss: 1.2877, Train Acc: 0.5268, Val Acc: 0.4949
Epoch: 110, Loss: 1.2927, Train Acc: 0.5491, Val Acc: 0.5053
Epoch: 120, Loss: 1.2637, Train Acc: 0.5100, Val Acc: 0.4720
Epoch: 130, Loss: 1.2730, Train Acc: 0.5702, Val Acc: 0.5211
Epoch: 140, Loss: 1.2782, Train Acc: 0.5465, Val Acc: 0.4928
Epoch: 150, Loss: 1.2693, Train Acc: 0.5748, Val Acc

## graphSAGE with MLP head

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Flickr
import torch_geometric.transforms as T
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import SAGEConv, BatchNorm

# 1. Load Dataset and apply benchmarking splits
transform = T.RandomNodeSplit(num_val=0.1, num_test=0.2)
dataset = Flickr(root='./data/Flickr', transform=transform)
data = dataset[0]

# 2. Setup NeighborLoader for efficient training
# GraphSAGE is famous for this inductive approach: sampling a fixed size neighborhood
train_loader = NeighborLoader(
    data,
    num_neighbors=[15, 10], 
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

# 3. GraphSAGE Architecture with MLP Head
class GraphSAGEWithMLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        
        # --- GNN Encoder (Feature Aggregation) ---
        # SAGEConv defaults to 'mean' aggregation
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)

        # --- MLP Task Head (Node-wise classification logic) ---
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(hidden_channels, hidden_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(hidden_channels, out_channels)
        )

    def forward(self, x, edge_index):
        # Encoder Phase
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        
        # Task Head Phase
        x = self.mlp(x)
        
        return x

# 4. Initialize on Local GPU (CUDA)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphSAGEWithMLP(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=dataset.num_classes
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=5e-4)

# 5. Training and Benchmarking Functions
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)
        # Loss calculated only for target nodes in the mini-batch
        loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.batch_size
    return total_loss / int(data.train_mask.sum())

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs

# 6. Execution Loop
print(f"Training GraphSAGE + MLP Head on {device}...")
best_val_acc = 0.0  # Track the highest validation accuracy achieved

for epoch in range(1, 201):
    loss = train()
    
    # We must calculate val_acc every epoch to find the 'best' moment
    train_acc, val_acc, test_acc = test()
    
    # Check if this is the best model so far
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        # Save the model state
        torch.save(model.state_dict(), 'graphSAGE_bestmodel(with MLP_head).pt')
        # Optional: Print to confirm save
        # print(f"--- Best model saved at Epoch {epoch:03d} (Val Acc: {val_acc:.4f}) ---")

    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, '
              f'Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

print(f"\nTraining Complete! Best Validation Accuracy: {best_val_acc:.4f}")

Training GraphSAGE + MLP Head on cuda...
Epoch: 010, Loss: 1.4676, Val Acc: 0.5063, Test Acc: 0.5045
Epoch: 020, Loss: 1.4136, Val Acc: 0.5157, Test Acc: 0.5106
Epoch: 030, Loss: 1.3827, Val Acc: 0.5170, Test Acc: 0.5155
Epoch: 040, Loss: 1.3554, Val Acc: 0.5200, Test Acc: 0.5194
Epoch: 050, Loss: 1.3277, Val Acc: 0.5191, Test Acc: 0.5186
Epoch: 060, Loss: 1.3032, Val Acc: 0.5231, Test Acc: 0.5228
Epoch: 070, Loss: 1.2766, Val Acc: 0.5217, Test Acc: 0.5230
Epoch: 080, Loss: 1.2549, Val Acc: 0.5219, Test Acc: 0.5221
Epoch: 090, Loss: 1.2300, Val Acc: 0.5182, Test Acc: 0.5159
Epoch: 100, Loss: 1.2112, Val Acc: 0.5141, Test Acc: 0.5150
Epoch: 110, Loss: 1.1932, Val Acc: 0.5141, Test Acc: 0.5156
Epoch: 120, Loss: 1.1795, Val Acc: 0.5154, Test Acc: 0.5154
Epoch: 130, Loss: 1.1634, Val Acc: 0.5073, Test Acc: 0.5164
Epoch: 140, Loss: 1.1433, Val Acc: 0.5054, Test Acc: 0.5168
Epoch: 150, Loss: 1.1276, Val Acc: 0.5060, Test Acc: 0.5124
Epoch: 160, Loss: 1.1149, Val Acc: 0.5067, Test Acc: 0.5123